# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

c:\Users\tae37\Desktop\실습\ML2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
grid_params = {'n_estimators':[100,200,300],'max_depth':[4,8,12]}
search_base = RandomForestRegressor(random_state=42,n_jobs=-1)
grid_search = GridSearchCV(search_base,grid_params,cv=5,scoring='r2',n_jobs=1)
start = time.perf_counter(); grid_search.fit(X_train,y_train); grid_seconds = time.perf_counter()-start
grid_mean = float(grid_search.cv_results_['mean_test_score'][grid_search.best_index_])
grid_std = float(grid_search.cv_results_['std_test_score'][grid_search.best_index_])
print('Grid best params:',grid_search.best_params_)
print('CV mean:',grid_mean,'std:',grid_std,'seconds:',grid_seconds)
display(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score','rank_test_score']])

Grid best params: {'max_depth': 12, 'n_estimators': 200}
CV mean: 0.8206179847869421 std: 0.06442538938888975 seconds: 5.90464989999964


,params,mean_test_score,std_test_score,rank_test_score
0,"{'max_depth': 4, 'n_estimators': 100}",0.779369,0.051308,9
1,"{'max_depth': 4, 'n_estimators': 200}",0.779586,0.049544,8
2,"{'max_depth': 4, 'n_estimators': 300}",0.780580,0.050523,7
3,"{'max_depth': 8, 'n_estimators': 100}",0.817761,0.065969,6
4,"{'max_depth': 8, 'n_estimators': 200}",0.817849,0.064449,5
5,"{'max_depth': 8, 'n_estimators': 300}",0.818850,0.063274,4
6,"{'max_depth': 12, 'n_estimators': 100}",0.820348,0.065524,2
7,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,1
8,"{'max_depth': 12, 'n_estimators': 300}",0.820182,0.065787,3


### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  
지정한 유한한 후보 조합을 모두 평가하여 그 격자 안의 최적 조합을 빠짐없이 찾습니다. 하지만 파라미터와 후보 수가 늘면 조합 수가 급증하고, 격자 사이의 값이나 격자 밖의 좋은 조합은 놓칩니다.

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [3]:
random_params = {'n_estimators':randint(100,301),'max_depth':randint(4,13)}
random_search = RandomizedSearchCV(search_base,random_params,n_iter=9,cv=5,scoring='r2',random_state=42,n_jobs=1)
start = time.perf_counter(); random_search.fit(X_train,y_train); random_seconds = time.perf_counter()-start
random_mean = float(random_search.cv_results_['mean_test_score'][random_search.best_index_])
random_std = float(random_search.cv_results_['std_test_score'][random_search.best_index_])
print('Random best params:',random_search.best_params_)
print('CV mean:',random_mean,'std:',random_std,'seconds:',random_seconds)
display(pd.DataFrame(random_search.cv_results_)[['params','mean_test_score','std_test_score','rank_test_score']])

Random best params: {'max_depth': 11, 'n_estimators': 230}
CV mean: 0.8210296383331107 std: 0.06508647306696069 seconds: 6.260735199997725


,params,mean_test_score,std_test_score,rank_test_score
0,"{'max_depth': 10, 'n_estimators': 279}",0.818594,0.069279,4
1,"{'max_depth': 11, 'n_estimators': 288}",0.820919,0.066358,2
2,"{'max_depth': 8, 'n_estimators': 202}",0.818300,0.063981,5
3,"{'max_depth': 6, 'n_estimators': 174}",0.809847,0.061638,8
4,"{'max_depth': 11, 'n_estimators': 216}",0.819858,0.065923,3
5,"{'max_depth': 7, 'n_estimators': 203}",0.816651,0.062739,7
6,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,1
7,"{'max_depth': 9, 'n_estimators': 152}",0.817948,0.067975,6
8,"{'max_depth': 5, 'n_estimators': 187}",0.801043,0.055673,9


### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  
탐색 차원이 크거나 일부 파라미터만 성능에 크게 영향을 주고 계산 예산이 제한된 경우입니다. n_iter로 평가 횟수를 정하고 넓은 공간을 탐색할 수 있습니다.

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [4]:
def objective(trial):
    params = {'n_estimators':trial.suggest_int('n_estimators',100,300),
              'max_depth':trial.suggest_int('max_depth',4,12)}
    trial_model = RandomForestRegressor(**params,random_state=42,n_jobs=-1)
    scores = cross_val_score(trial_model,X_train,y_train,cv=5,scoring='r2',n_jobs=1)
    trial.set_user_attr('cv_std',float(scores.std()))
    return float(scores.mean())
sampler = optuna.samplers.TPESampler(seed=42,n_startup_trials=3)
study = optuna.create_study(direction='maximize',sampler=sampler)
start = time.perf_counter(); study.optimize(objective,n_trials=9); optuna_seconds = time.perf_counter()-start
print('Optuna best params:',study.best_params,'best CV mean:',study.best_value)
print('Search seconds:',optuna_seconds)
display(study.trials_dataframe()[['number','value','params_n_estimators','params_max_depth']])

[I 2026-10-07 16:43:43,974] A new study created in memory with name: no-name-e507e1d6-51a4-429f-bf4f-c6615ae9c35c
[I 2026-10-07 16:43:44,593] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 16:43:45,387] Trial 1 finished with value: 0.8184595590761907 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 16:43:45,890] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 16:43:46,346] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-07 16:43:47,040] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

Optuna best params: {'n_estimators': 175, 'max_depth': 12} best CV mean: 0.8212542415716282
Search seconds: 5.963277499999094


,number,value,params_n_estimators,params_max_depth
0,0,0.821254,175,12
1,1,0.818460,247,9
2,2,0.799029,131,5
3,3,0.820148,109,12
4,4,0.820595,204,12
5,5,0.779559,274,4
6,6,0.818049,193,8
7,7,0.816309,109,7
8,8,0.818672,290,10


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
이번 Optuna TPE는 초기 3개 trial 이후 이전 후보의 파라미터와 성능을 이용해 유망한 다음 후보를 제안합니다. Grid·Random은 이전 점수에 따라 후보 제안을 바꾸지 않습니다. TPE도 전역 최적값을 보장하지는 않습니다.

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [5]:
optuna_model = RandomForestRegressor(**study.best_params,random_state=42,n_jobs=-1)
optuna_scores = cross_val_score(optuna_model,X_train,y_train,cv=5,scoring='r2',n_jobs=1)
optuna_mean = float(optuna_scores.mean()); optuna_std = float(optuna_scores.std())
print('Fold scores:',optuna_scores)
print('CV mean:',optuna_mean,'std:',optuna_std)

Fold scores: [0.86941442 0.69669635 0.82799429 0.84277018 0.86939597]
CV mean: 0.8212542415716282 std: 0.06428073023916543


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [6]:
from sklearn.model_selection import KFold,cross_validate
import numpy as np
from pathlib import Path
from datetime import timezone
search_comparison = pd.DataFrame([
 {'search_method':'Grid','best_params':grid_search.best_params_,'cv_mean':grid_mean,'cv_std':grid_std,'search_seconds':grid_seconds},
 {'search_method':'Random','best_params':random_search.best_params_,'cv_mean':random_mean,'cv_std':random_std,'search_seconds':random_seconds},
 {'search_method':'Optuna','best_params':study.best_params,'cv_mean':optuna_mean,'cv_std':optuna_std,'search_seconds':optuna_seconds}])
display(search_comparison)
# CV 평균을 최우선으로 선택합니다. 근접 후보는 표준편차·시간도 함께 표시합니다.
chosen_search = search_comparison.sort_values(['cv_mean','cv_std','search_seconds'],ascending=[False,True,True]).iloc[0]
candidate_params = dict(chosen_search['best_params'])
baseline_params = {'n_estimators':100,'max_depth':None}
comparison_cv = KFold(n_splits=5,shuffle=False)
comparison_models = {
 'Baseline':RandomForestRegressor(**baseline_params,random_state=42,n_jobs=-1),
 'Tuned':RandomForestRegressor(**candidate_params,random_state=42,n_jobs=-1)}
cv_details = {}
for name,model in comparison_models.items():
    cv_details[name] = cross_validate(model,X_train,y_train,cv=comparison_cv,scoring='r2',return_train_score=True,n_jobs=1)
fold_comparison = pd.DataFrame({'fold':range(1,6),
 'baseline_train_r2':cv_details['Baseline']['train_score'],
 'baseline_validation_r2':cv_details['Baseline']['test_score'],
 'tuned_train_r2':cv_details['Tuned']['train_score'],
 'tuned_validation_r2':cv_details['Tuned']['test_score']})
tuning_rows = []
for name,result in cv_details.items():
    train_mean = float(result['train_score'].mean()); valid_mean = float(result['test_score'].mean())
    tuning_rows.append({'model':name,'train_mean':train_mean,'cv_mean':valid_mean,
        'cv_std':float(result['test_score'].std()),'train_valid_gap':train_mean-valid_mean})
tuning_comparison = pd.DataFrame(tuning_rows).set_index('model')
tuning_delta = float(tuning_comparison.loc['Tuned','cv_mean']-tuning_comparison.loc['Baseline','cv_mean'])
display(fold_comparison); display(tuning_comparison)
print('튜닝 전후 CV R2 차이:',tuning_delta)
final_choice = 'Tuned' if tuning_delta>0 else 'Baseline'
final_params = candidate_params if final_choice=='Tuned' else baseline_params
final_search_method = str(chosen_search['search_method']) if final_choice=='Tuned' else 'Baseline (no search)'
print('Test 확인 전 최종 선택 확정:',final_choice,final_params)
final_model = RandomForestRegressor(**final_params,random_state=42,n_jobs=-1).fit(X_train,y_train)
final_test_r2 = float(final_model.score(X_test,y_test))  # hold-out Test 평가 1회
final_cv_mean = float(tuning_comparison.loc[final_choice,'cv_mean'])
final_cv_std = float(tuning_comparison.loc[final_choice,'cv_std'])
cv_test_gap = final_cv_mean-final_test_r2
print('최종 Test R2:',final_test_r2,'CV mean − Test R2:',cv_test_gap)
artifact_dir = Path('model_artifacts'); artifact_dir.mkdir(exist_ok=True)
joblib.dump(final_model,artifact_dir/'best_model.pkl')
model_meta = {'model_name':type(final_model).__name__,'saved_at':datetime.now(timezone.utc).isoformat(),
 'selected_search_method':final_search_method,'best_params':final_params,'cv_mean':final_cv_mean,'cv_std':final_cv_std,
 'final_test_r2':final_test_r2,'baseline_settings':baseline_params,'tuned_candidate_settings':candidate_params,
 'tuned_candidate_search_method':str(chosen_search['search_method']),
 'tuning_cv_comparison':tuning_comparison.reset_index().to_dict(orient='records'),
 'tuning_cv_r2_delta':tuning_delta,'cv_mean_minus_test_r2':cv_test_gap,'feature_order':features,
 'random_state':42,'cv':'KFold(n_splits=5, shuffle=False)'}
(artifact_dir/'model_meta.json').write_text(json.dumps(model_meta,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 완료: model_artifacts/best_model.pkl, model_artifacts/model_meta.json')

,search_method,best_params,cv_mean,cv_std,search_seconds
0,Grid,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,5.904650
1,Random,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,6.260735
2,Optuna,"{'n_estimators': 175, 'max_depth': 12}",0.821254,0.064281,5.963277


,fold,baseline_train_r2,baseline_validation_r2,tuned_train_r2,tuned_validation_r2
0,1,0.971728,0.862667,0.971451,0.869414
1,2,0.975444,0.680790,0.976382,0.696696
2,3,0.974969,0.834270,0.972136,0.827994
3,4,0.974586,0.845045,0.972621,0.842770
4,5,0.971663,0.872282,0.970872,0.869396


,train_mean,cv_mean,cv_std,train_valid_gap
model,,,,
Baseline,0.973678,0.819011,0.070370,0.154667
Tuned,0.972692,0.821254,0.064281,0.151438


튜닝 전후 CV R2 차이: 0.0022435439729445728
Test 확인 전 최종 선택 확정: Tuned {'n_estimators': 175, 'max_depth': 12}
최종 Test R2: 0.8910932915341476 CV mean − Test R2: -0.06983904996251944
저장 완료: model_artifacts/best_model.pkl, model_artifacts/model_meta.json


### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
무조건 선택할 필요는 없습니다. 작은 평균 차이가 Fold 변동보다 작다면 안정성, 탐색·재학습 비용, 운영 요구를 함께 판단합니다. 이번 코드는 CV 평균을 최우선으로 선택하고 표준편차와 시간도 보고하며, 작은 차이를 확실한 우월성으로 단정하지 않습니다.

### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  
Test를 보며 후보나 설정을 바꾸면 Test도 선택 과정에 들어가 일반화 성능 추정이 낙관적으로 편향됩니다. 모든 선택을 Train 내부 CV로 끝낸 뒤 최종 모델만 Test에서 한 번 확인해야 합니다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**
튜닝 후보는 Optuna의 {'n_estimators': 175, 'max_depth': 12}입니다. CV 평균 R²는 0.819011→0.821254, 차이는 +0.002244입니다(백분율이 아님). 표준편차는 0.070370→0.064281, Train−검증 차이는 0.154667→0.151438입니다. 동일한 5개 Fold에서 비교했고, CV 평균이 개선되어 튜닝 후보를 최종 선택했습니다. Fold의 변동과 작은 차이도 함께 고려해야 하며 이번 CV는 선택에 사용되었으므로 독립 평가가 아닙니다.

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**
최종 선택은 Tuned이고 CV 평균 R²=0.821254, Test R²=0.891093, CV−Test=-0.069839입니다. Test는 선택 확정 후 한 번 평가했습니다. Train 점수가 검증보다 높아 과적합 가능성은 남지만, CV/Test 차이만으로 과적합을 확정할 수 없습니다. 분할 난이도와 표본 변동, CV 후보 선택의 낙관 편향도 영향을 줍니다. 점수가 비슷하거나 Test가 더 높다고 일반화가 보장되지는 않으며 Test 이후 설정·후보를 바꾸지 않았습니다.


---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?  
-> Grid는 정한 격자의 모든 조합, Random은 정한 횟수의 무작위 후보를 평가합니다.

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?  
-> 평가할 후보 조합 수입니다.

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?  
-> study는 탐색 전체, trial은 한 후보 평가입니다.

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?  
-> Fold 표준편차, 학습·탐색 비용과 운영 목적입니다.

5. hold-out Test는 언제 사용해야 하나요?  
-> 최종 선택을 확정한 뒤 한 번 사용합니다.

6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?  
-> 모델 재사용, 특성 순서와 선택 근거 추적, 재현과 배포를 위해 저장합니다.